<a href="https://colab.research.google.com/github/yuzxin/Capstone/blob/main/%EC%95%84%EA%B9%8C%EC%8B%9C%EB%82%98%EB%AC%B4_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("내 컴퓨터에 있는 아까시나무 파일을 선택해주세요.")
uploaded = files.upload() # 실행하면 [파일 선택] 버튼이 뜹니다.

내 컴퓨터에 있는 아까시나무 파일을 선택해주세요.


Saving OBS_계절관측_아까시나무.xlsx to OBS_계절관측_아까시나무 (1).xlsx


In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import files

# ==========================================
# 1. 데이터 파일 로드
file_path = "OBS_계절관측_아까시나무.xlsx"

# 아까시나무 원자료는 앞 2행이 다중 헤더이므로 header=None으로 불러옴
raw = pd.read_excel(file_path, header=None)

print("원본 데이터 크기:", raw.shape)
display(raw.head())

원본 데이터 크기: (229, 8)


,0,1,2,3,4,5,6,7
0,지점,년도,아까시나무,아까시나무,아까시나무,아까시나무,아까시나무,아까시나무
1,NaN,NaN,발아,발아(평비),개화,개화(평비),만발,만발(평비)
2,북강릉,2016,2016-04-15 00:00:00,-8일,2016-05-06 00:00:00,-5일,2016-05-09 00:00:00,-6일
3,북강릉,2017,2017-04-12 00:00:00,-11일,2017-05-05 00:00:00,-6일,2017-05-07 00:00:00,-8일
4,북강릉,2018,2018-04-16 00:00:00,-7일,2018-05-01 00:00:00,-10일,2018-05-04 00:00:00,-11일


In [ ]:

# ==========================================
# 2. 컬럼 정리 및 실제 관측값 추출
# ==========================================
# 앞 2행 제거 후 실제 관측값만 사용
df = raw.iloc[2:].reset_index(drop=True)

# 앞의 8개 컬럼만 사용
df = df.iloc[:, :8]

# 컬럼명 표준화 (벚나무 구조와 동일하게 통일)
df.columns = [
    "지점",
    "년도",
    "발아일",
    "발아_평비",
    "개화일",
    "개화_평비",
    "만발일",
    "만발_평비"
]

# 빈 행 제거
df = df.dropna(how="all").reset_index(drop=True)

# 지점명 공백 제거
df["지점"] = df["지점"].astype(str).str.strip()

# 아까시나무 특유의 결측 기호들을 파이썬 NaN으로 일괄 변경
df = df.replace(['관측 안됨', '―', '', ' ', 'None', 'nan'], np.nan)

print("정리 후 데이터 크기:", df.shape)
display(df.head())


정리 후 데이터 크기: (227, 8)


/tmp/ipykernel_25328/2329066688.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(['관측 안됨', '―', '', ' ', 'None', 'nan'], np.nan)


,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비
0,북강릉,2016,2016-04-15,-8일,2016-05-06,-5일,2016-05-09,-6일
1,북강릉,2017,2017-04-12,-11일,2017-05-05,-6일,2017-05-07,-8일
2,북강릉,2018,2018-04-16,-7일,2018-05-01,-10일,2018-05-04,-11일
3,북강릉,2019,2019-05-08,15일,2019-05-12,1일,2019-05-15,0일
4,북강릉,2020,2020-04-29,6일,2020-05-10,-1일,2020-05-14,-1일


In [ ]:
# ==========================================
# 3. 자료형 변환
# ==========================================
# 년도 숫자형 변환
df["년도"] = pd.to_numeric(df["년도"], errors="coerce")

# 날짜형 변환
for col in ["발아일", "개화일", "만발일"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 평년비 숫자형 변환
# 예: "-8일", "15일", "0일" -> -8, 15, 0
for col in ["발아_평비", "개화_평비", "만발_평비"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("일", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip()
        .replace(["nan", "", "-", "None"], np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("자료형 변환 완료")
print(df.dtypes)
display(df.head())



자료형 변환 완료
지점               object
년도                int64
발아일      datetime64[ns]
발아_평비           float64
개화일      datetime64[ns]
개화_평비           float64
만발일      datetime64[ns]
만발_평비           float64
dtype: object


,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비
0,북강릉,2016,2016-04-15,-8.0,2016-05-06,-5.0,2016-05-09,-6.0
1,북강릉,2017,2017-04-12,-11.0,2017-05-05,-6.0,2017-05-07,-8.0
2,북강릉,2018,2018-04-16,-7.0,2018-05-01,-10.0,2018-05-04,-11.0
3,북강릉,2019,2019-05-08,15.0,2019-05-12,1.0,2019-05-15,0.0
4,북강릉,2020,2020-04-29,6.0,2020-05-10,-1.0,2020-05-14,-1.0


In [ ]:
# ==========================================
# 4. 결측치 확인
# ==========================================
missing = df.isna().sum().reset_index()
missing.columns = ["컬럼", "결측치_개수"]
missing["결측치_비율(%)"] = (
    missing["결측치_개수"] / len(df) * 100
).round(2)

display(missing)
print("전체 결측치 총합:", df.isna().sum().sum())




,컬럼,결측치_개수,결측치_비율(%)
0,지점,0,0.00
1,년도,0,0.00
2,발아일,3,1.32
3,발아_평비,12,5.29
4,개화일,3,1.32
5,개화_평비,12,5.29
6,만발일,4,1.76
7,만발_평비,87,38.33


전체 결측치 총합: 121


In [ ]:
# ==========================================
# 5. 날짜 분석용 / 평년비 분석용 데이터 분리
# ==========================================
date_required_cols = ["지점", "년도", "발아일", "개화일", "만발일"]
anomaly_cols = ["발아_평비", "개화_평비", "만발_평비"]

# 날짜 핵심 컬럼 결측 확인
date_missing_rows = df[df[date_required_cols].isna().any(axis=1)]

print("날짜 핵심 컬럼 결측 행 수:", len(date_missing_rows))
display(date_missing_rows)

# [분리 1] 날짜 분석용 데이터 (날짜 3개가 모두 존재할 때 유지)
df_date = df.dropna(subset=date_required_cols).copy()
df_date["년도"] = df_date["년도"].astype(int)

# 평비 결측 여부 표시
for col in anomaly_cols:
    df_date[col + "_결측여부"] = df_date[col].isna()

# [분리 2] 평년비 분석용 데이터 (날짜가 있고, 평비도 결측이 없을 때)
df_anomaly = df_date.dropna(subset=anomaly_cols).copy()

print("날짜 분석용 데이터 크기:", df_date.shape)
print("평년비 분석용 데이터 크기:", df_anomaly.shape)

print("평비 결측으로 평년비 분석에서 제외될 행")
display(df_date[df_date[anomaly_cols].isna().any(axis=1)])




날짜 핵심 컬럼 결측 행 수: 4


,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비
19,춘천,2016,2016-04-15,NaN,2016-05-08,NaN,NaT,NaN
190,서귀포,2019,NaT,NaN,NaT,NaN,NaT,NaN
191,서귀포,2020,NaT,NaN,NaT,NaN,NaT,NaN
195,서귀포,2024,NaT,NaN,NaT,NaN,NaT,NaN


날짜 분석용 데이터 크기: (223, 11)
평년비 분석용 데이터 크기: (140, 11)
평비 결측으로 평년비 분석에서 제외될 행


,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비,발아_평비_결측여부,개화_평비_결측여부,만발_평비_결측여부
10,북춘천,2017,2017-05-04,8.0,2017-05-08,-6.0,2017-05-12,NaN,False,False,True
11,북춘천,2018,2018-04-26,0.0,2018-05-05,-9.0,2018-05-10,NaN,False,False,True
12,북춘천,2019,2019-04-27,1.0,2019-05-15,1.0,2019-05-21,NaN,False,False,True
13,북춘천,2020,2020-05-01,5.0,2020-05-12,-2.0,2020-05-18,NaN,False,False,True
14,북춘천,2021,2021-05-08,12.0,2021-05-12,-2.0,2021-05-14,NaN,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...
212,홍성,2021,2021-04-14,NaN,2021-05-11,NaN,2021-05-14,NaN,True,True,True
213,홍성,2022,2022-04-19,NaN,2022-05-10,NaN,2022-05-13,NaN,True,True,True
214,홍성,2023,2023-04-19,NaN,2023-05-04,NaN,2023-05-09,NaN,True,True,True
215,홍성,2024,2024-04-12,NaN,2024-05-07,NaN,2024-05-10,NaN,True,True,True


In [ ]:
# ==========================================
# 6. DOY(Day of Year) 생성
# ==========================================
for data in [df_date, df_anomaly]:
    data["발아_DOY"] = data["발아일"].dt.dayofyear
    data["개화_DOY"] = data["개화일"].dt.dayofyear
    data["만발_DOY"] = data["만발일"].dt.dayofyear

display(df_date.head())
display(df_anomaly.head())




,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비,발아_평비_결측여부,개화_평비_결측여부,만발_평비_결측여부,발아_DOY,개화_DOY,만발_DOY
0,광주,2016,2016-03-25,-12.0,2016-04-22,-11.0,2016-04-29,-8.0,False,False,False,85,113,120
1,대구,2016,2016-04-01,-5.0,2016-04-20,-13.0,2016-04-28,-10.0,False,False,False,92,111,119
2,대전,2016,2016-04-05,-9.0,2016-05-02,-5.0,2016-05-07,-5.0,False,False,False,96,123,128
3,목포,2016,2016-04-08,-5.0,2016-04-29,-8.0,2016-05-07,-5.0,False,False,False,99,120,128
4,백령도,2016,2016-05-06,2.0,2016-05-26,1.0,2016-05-31,NaN,False,False,True,127,147,152


,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비,발아_평비_결측여부,개화_평비_결측여부,만발_평비_결측여부,발아_DOY,개화_DOY,만발_DOY
0,광주,2016,2016-03-25,-12.0,2016-04-22,-11.0,2016-04-29,-8.0,False,False,False,85,113,120
1,대구,2016,2016-04-01,-5.0,2016-04-20,-13.0,2016-04-28,-10.0,False,False,False,92,111,119
2,대전,2016,2016-04-05,-9.0,2016-05-02,-5.0,2016-05-07,-5.0,False,False,False,96,123,128
3,목포,2016,2016-04-08,-5.0,2016-04-29,-8.0,2016-05-07,-5.0,False,False,False,99,120,128
4,북강릉,2016,2016-04-15,-8.0,2016-05-06,-5.0,2016-05-09,-6.0,False,False,False,106,127,130


In [ ]:
# ==========================================
# 7. 날짜 순서 오류 확인 및 제거
# ==========================================
# 기준: 발아일 ≤ 개화일 ≤ 만발일
logic_error_date = df_date[
    (df_date["개화일"] < df_date["발아일"]) |
    (df_date["만발일"] < df_date["개화일"]) |
    (df_date["만발일"] < df_date["발아일"])
]

logic_error_anomaly = df_anomaly[
    (df_anomaly["개화일"] < df_anomaly["발아일"]) |
    (df_anomaly["만발일"] < df_anomaly["개화일"]) |
    (df_anomaly["만발일"] < df_anomaly["발아일"])
]

print("날짜 분석용 날짜 순서 오류:", len(logic_error_date))
display(logic_error_date)

print("평년비 분석용 날짜 순서 오류:", len(logic_error_anomaly))
display(logic_error_anomaly)

# 오류 제거
df_date = df_date[
    ~(
        (df_date["개화일"] < df_date["발아일"]) |
        (df_date["만발일"] < df_date["개화일"]) |
        (df_date["만발일"] < df_date["발아일"])
    )
].copy()

df_anomaly = df_anomaly[
    ~(
        (df_anomaly["개화일"] < df_anomaly["발아일"]) |
        (df_anomaly["만발일"] < df_anomaly["개화일"]) |
        (df_anomaly["만발일"] < df_anomaly["발아일"])
    )
].copy()




In [ ]:
# ==========================================
# 8. DOY 이상치 확인 및 제거
# ==========================================
# (발아: 4~5월 / 개화·만발: 5~6월 분포 반영)
doy_outlier_date = df_date[
    (df_date["발아_DOY"] < 50) | (df_date["발아_DOY"] > 150) |   # 약 2월 중순 ~ 5월 말
    (df_date["개화_DOY"] < 70) | (df_date["개화_DOY"] > 170) |   # 약 3월 중순 ~ 6월 중순
    (df_date["만발_DOY"] < 75) | (df_date["만발_DOY"] > 180)     # 약 3월 말 ~ 6월 말
]

doy_outlier_anomaly = df_anomaly[
    (df_anomaly["발아_DOY"] < 50) | (df_anomaly["발아_DOY"] > 150) |
    (df_anomaly["개화_DOY"] < 70) | (df_anomaly["개화_DOY"] > 170) |
    (df_anomaly["만발_DOY"] < 75) | (df_anomaly["만발_DOY"] > 180)
]

print("날짜 분석용 DOY 이상치:", len(doy_outlier_date))
display(doy_outlier_date)

print("평년비 분석용 DOY 이상치:", len(doy_outlier_anomaly))
display(doy_outlier_anomaly)

# 이상치 제거
df_date = df_date[
    (df_date["발아_DOY"].between(50, 150)) &
    (df_date["개화_DOY"].between(70, 170)) &
    (df_date["만발_DOY"].between(75, 180))
].copy()

df_anomaly = df_anomaly[
    (df_anomaly["발아_DOY"].between(50, 150)) &
    (df_anomaly["개화_DOY"].between(70, 170)) &
    (df_anomaly["만발_DOY"].between(75, 180))
].copy()




In [ ]:
# ==========================================
# 9. 중복 확인
# ==========================================
dup_date = df_date[df_date.duplicated(subset=["지점", "년도"], keep=False)]
dup_anomaly = df_anomaly[df_anomaly.duplicated(subset=["지점", "년도"], keep=False)]

print("날짜 분석용 중복 행 수:", len(dup_date))
display(dup_date)

print("평년비 분석용 중복 행 수:", len(dup_anomaly))
display(dup_anomaly)




In [ ]:
# ==========================================
# 10. 연도별 정렬
# ==========================================
df_date = df_date.sort_values(by=["년도", "지점"]).reset_index(drop=True)
df_anomaly = df_anomaly.sort_values(by=["년도", "지점"]).reset_index(drop=True)


# ==========================================
# 11. 최종 점검
# ==========================================
print("===== 아까시나무 날짜 분석용 최종 데이터 =====")
print("데이터 크기:", df_date.shape)
print("연도 범위:", df_date["년도"].min(), "~", df_date["년도"].max())
print("지점 수:", df_date["지점"].nunique())
display(df_date.head())

print("===== 아까시나무 평년비 분석용 최종 데이터 =====")
print("데이터 크기:", df_anomaly.shape)
print("연도 범위:", df_anomaly["년도"].min(), "~", df_anomaly["년도"].max())
print("지점 수:", df_anomaly["지점"].nunique())
display(df_anomaly.head())


# ==========================================
# 12. CSV 저장 및 다운로드
# ==========================================
df_date.to_csv("acacia_date_clean.csv", index=False, encoding="utf-8-sig")
df_anomaly.to_csv("acacia_anomaly_clean.csv", index=False, encoding="utf-8-sig")

print("CSV 저장 및 다운로드 시작...")
files.download("acacia_date_clean.csv")
files.download("acacia_anomaly_clean.csv")

,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비,발아_평비_결측여부,개화_평비_결측여부,만발_평비_결측여부,발아_DOY,개화_DOY,만발_DOY
0,북강릉,2016,2016-04-15,-8.0,2016-05-06,-5.0,2016-05-09,-6.0,False,False,False,106,127,130
1,북강릉,2017,2017-04-12,-11.0,2017-05-05,-6.0,2017-05-07,-8.0,False,False,False,102,125,127
2,북강릉,2018,2018-04-16,-7.0,2018-05-01,-10.0,2018-05-04,-11.0,False,False,False,106,121,124
3,북강릉,2019,2019-05-08,15.0,2019-05-12,1.0,2019-05-15,0.0,False,False,False,128,132,135
4,북강릉,2020,2020-04-29,6.0,2020-05-10,-1.0,2020-05-14,-1.0,False,False,False,120,131,135


,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비,발아_평비_결측여부,개화_평비_결측여부,만발_평비_결측여부,발아_DOY,개화_DOY,만발_DOY
0,북강릉,2016,2016-04-15,-8.0,2016-05-06,-5.0,2016-05-09,-6.0,False,False,False,106,127,130
1,북강릉,2017,2017-04-12,-11.0,2017-05-05,-6.0,2017-05-07,-8.0,False,False,False,102,125,127
2,북강릉,2018,2018-04-16,-7.0,2018-05-01,-10.0,2018-05-04,-11.0,False,False,False,106,121,124
3,북강릉,2019,2019-05-08,15.0,2019-05-12,1.0,2019-05-15,0.0,False,False,False,128,132,135
4,북강릉,2020,2020-04-29,6.0,2020-05-10,-1.0,2020-05-14,-1.0,False,False,False,120,131,135


날짜 분석용 날짜 순서 오류: 0


,지점,년도,발아일,발아_평비,개화일,개화_평비,만발일,만발_평비,발아_평비_결측여부,개화_평비_결측여부,만발_평비_결측여부,발아_DOY,개화_DOY,만발_DOY


평년비 분석용 날짜 순서 오류: 0


...

CSV 저장 및 다운로드 시작...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>